In [39]:
import numpy as np
from orion.utils.o3d_utils import visualize_o3d_point_cloud, O3DPointCloud, estimate_rotation, draw_pose_axes, create_o3d_from_points_and_color
from orion.utils.misc_utils import plotly_draw_3d_pcd, transform_points, create_point_clouds_from_keypoints, to_homo
import open3d as o3d
import plotly.graph_objects as go
import os
import json
import h5py
import torch
from PIL import Image
import cv2
import trimesh
#from scripts.extension_work.run_dust3r_on_video import get_intrinsics

from enum import Enum, auto


In [40]:
class DepthType(Enum):
    GT = auto()
    DUST3R = auto()
    METRIC3D = auto()
    DEPTHCRAFTER = auto()
    DEPTHPRO = auto()

In [41]:
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/salt_distract5_demo"
foundation_pose_dir = os.path.join(annotation_folder, "FoundationPose")

In [ ]:
demo_name = annotation_folder.split("/")[-1]
with open(os.path.join(annotation_folder, "config.json"), "r") as f:
    config_dict = json.load(f)
demo_file = h5py.File(config_dict["original_file"], "r")

seq_len = demo_file["data/human_demo/obs/agentview_rgb"][()].shape[0]
subsequence_range = [0, seq_len]
#has_depth = "data/human_demo/obs/agentview_depth/" in demo_file
# gt_depth = False
# dust3r_depth = True
# metric_est_depth = False

depth_type = DepthType.METRIC3D

if depth_type == DepthType.GT:
    depths = demo_file["data/human_demo/obs/agentview_depth/"][()][subsequence_range[0]:subsequence_range[1]]
    # cotracker_dict = torch.load(os.path.join(annotation_folder, "tracked_points.pt"), map_location=torch.device('cuda:3'))
    # trajs_2d = cotracker_dict['pred_tracks'].cpu().detach().numpy()[0]
    # trajs_2d = trajs_2d[subsequence_range[0]:subsequence_range[1]]
elif depth_type == DepthType.DUST3R:
    depths = demo_file["data/human_demo/obs/dust3r_depth"][()][subsequence_range[0]:subsequence_range[1]]
    depths *= 1000
elif depth_type == DepthType.METRIC3D:
    #check if file exists
    # if os.path.isfile(os.path.join(annotation_folder, "metric3d", "all_pred_depth_interp_nearest.npy")):    
    #     depths = np.load(os.path.join(annotation_folder, "metric3d", "all_pred_depth_interp_nearest.npy"))
    # else:
    #     raise Exception("No metric 3d depth file found")
    
    if os.path.isfile(os.path.join(foundation_pose_dir, "depth", "all_pred_depth.npy")):    
        depths = np.load(os.path.join(foundation_pose_dir, "depth", "all_pred_depth.npy"))
    else:
        raise Exception("No metric 3d depth file found")
elif depth_type == DepthType.DEPTHPRO:
    depths = np.load(os.path.join(annotation_folder, "DepthPro", "depth_interp_bilinear.npy"))
    depths *= 1000
elif depth_type == DepthType.DEPTHCRAFTER:
    depths = np.load(os.path.join(annotation_folder, "DepthCrafter", "depth_crafter_output.npz"))['depth']
    #depths *= 1000
    depths = (depths.min() + (depths.max() - depths.min()) * (1 - depths))
    

data_config = json.loads(demo_file["data"].attrs['data_config'])
K = np.zeros((3, 3))
K[0,0] = data_config["intrinsics"]["camera_rs_0"]["fx"]
K[1,1] = data_config["intrinsics"]["camera_rs_0"]["fy"]
K[0,2] = data_config["intrinsics"]["camera_rs_0"]["cx"]
K[1,2] = data_config["intrinsics"]["camera_rs_0"]["cy"]
K[2,2] = 1

print("original K", K)

if depth_type == DepthType.DUST3R:
    original_images = demo_file["data/human_demo/obs/agentview_rgb/"][()][subsequence_range[0]:subsequence_range[1]]
    images = demo_file["data/human_demo/obs/dust3r_rgb/"][()][subsequence_range[0]:subsequence_range[1]]
    images = (images * (255 / images.max())).astype(np.uint8)
    #K = dust3r_intrinsics[0]

    # since dust3r downsamples the image, we need to adjust K
    y_scale, x_scale =  images.shape[1] / original_images.shape[1], images.shape[2] / original_images.shape[2]
    K[0,0] *= x_scale
    K[1,1] *= y_scale
    K[0,2] *= x_scale
    K[1,2] *= y_scale
    
elif depth_type == DepthType.DEPTHCRAFTER:
    # resize images to depth shape
    original_images = demo_file["data/human_demo/obs/agentview_rgb/"][()][subsequence_range[0]:subsequence_range[1]]
    images = np.zeros((original_images.shape[0], depths.shape[1], depths.shape[2], 3), dtype=np.uint8)
    for i in range(images.shape[0]):
        images[i] = cv2.resize(original_images[i], (depths.shape[2], depths.shape[1]), interpolation=cv2.INTER_NEAREST)
    
    #adjust K
    y_scale, x_scale =  images.shape[1] / original_images.shape[1], images.shape[2] / original_images.shape[2]
    K[0,0] *= x_scale
    K[1,1] *= y_scale
    K[0,2] *= x_scale
    K[1,2] *= y_scale
else:
    images = demo_file["data/human_demo/obs/agentview_rgb/"][()][subsequence_range[0]:subsequence_range[1]]


print("New K", K)

print("Depth shape", depths.shape)
print("Depth range", np.min(depths), np.max(depths))

masks_orig = np.load(os.path.join(annotation_folder, 'masks.npz'))
masks_orig = masks_orig['arr_0']
masks_orig = masks_orig[subsequence_range[0]:subsequence_range[1]]
demo_file.close()

In [ ]:
# keypoints = []
# for i in range(images[0].shape[0]):
#     for j in range(images[0].shape[1]):
#             keypoints.append(np.asarray([j,i]))
# keypoints = np.asarray(keypoints)
# points_3d = create_point_clouds_from_keypoints(keypoints, depths[0], K, scaling_factor=1000)

frame_idx = 0
pcds = []
frame_indices = []
num_pcds = 3
for i in range(num_pcds):
    frame_idx = min((images.shape[0] // (num_pcds-1)) * i, images.shape[0]-1)
    print(frame_idx)
    pcd_wrapper = O3DPointCloud()
    pcd_wrapper.create_from_rgbd(images[frame_idx], depths[frame_idx], K, depth_trunc=3)
    pcd_wrapper.max_points = 40000
    pcd_wrapper.preprocess()
    pcds.append(pcd_wrapper)

# frame_idx = min((images.shape[0] // (num_pcds-1)) * i, images.shape[0]-1)
# pcd_wrapper = O3DPointCloud()

# #truncate the last 70% of the depth
# #get the cutoff value for truncation
# depths_trunc = np.sort(depths[frame_idx].flatten())
# depth_cutoff = depths_trunc[int(depths_trunc.shape[0] * 0.01)]


# pcd_wrapper.create_from_rgbd(images[frame_idx], depths[frame_idx], K, depth_trunc=depth_cutoff)
# pcd_wrapper.max_points = 40000
# pcd_wrapper.preprocess()
# pcds = [pcd_wrapper]

In [ ]:
all_points = []
all_colors = []
pcds = [pcds[0]]
for idx, pcd in enumerate(pcds):
    all_points.append(pcd.get_points())
    all_colors.append(pcd.get_colors())
    
    #append a different color for each frame
    # col = np.zeros_like(pcd.get_colors())
    # col[:, idx] = 255
    #all_colors.append(col)
    
all_points = np.concatenate(all_points, axis=0)
all_colors = np.concatenate(all_colors, axis=0)

temporal_pcd = O3DPointCloud()
temporal_pcd.pcd = create_o3d_from_points_and_color(all_points, all_colors)
#temporal_pcd.max_points = 40000
#temporal_pcd.preprocess()

save_folder = f"/home/arisrei/research/RPL/ORION/visualization/{demo_name}"
os.makedirs(save_folder, exist_ok=True)
vis_name = f"{demo_name}_{depth_type.name}_interp_bilinear.html"
print(f"Visualization saved at {os.path.join(save_folder, vis_name)}")

fig = plotly_draw_3d_pcd(temporal_pcd.get_points(), temporal_pcd.get_colors(), offline=True, additional_point_draw_lines=True, equal_axis=True, addition_points=None)
fig.write_html(os.path.join(save_folder, vis_name))
display(fig)


